This notebook must be executed inside the FINN container provided by Xilinx.

Follow the steps of the [FINN docks](https://finn.readthedocs.io/en/latest/), section [Quickstart](https://finn.readthedocs.io/en/latest/getting_started.html#running-finn-in-docker), to download, build and verify the container installation.

You must also move your project folder to inside the same folder the repository is located.

To start the container, go to the folder where the repo was installed and run $ ./run-docker.sh notebook

If you are using vscode, you can select the notebook kernel inside the container to run the code.

This notebook is strongly based on Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook and 0BAB1 [2_finn_hardware_layers.ipynb](https://github.com/0BAB1/tutorial-snippets/blob/main/8%20Python%20to%20FPGA/2_finn_hardware_layers.ipynb) notebook.

Check them out for deeper instructions.

# Setup Python Paths for Libraries

In [3]:
import os
import sys

# Correct the path where the environment starts to be the same folder of the notebook, so that the imports work correctly.
# When starting the container with the notebook server, the script hardcodes the Jupyter server to start inside the ./notebooks folder.
os.chdir('../QFast-SCNN_with_Brevitas_and_FINN/finn_environment')

# Add the train_environment directory to the system path to allow imports from there
sys.path.append(os.path.abspath('../train_environment'))
print(sys.path)

FileNotFoundError: [Errno 2] No such file or directory: '../QFast-SCNN_with_Brevitas_and_FINN/finn_environment'

# Setup Model for FINN

* Tidy up (and also after EACH step)
* Pre (data feed) / Post proc (top k)
* Model streamlining (Main step) + smaller example
* Model HW Layers (Generates Matrix Vector Activation Units for fc layers)
* Model data flow partitions (Generate a sub-graph for all HW convertible nodes)
* Specialize layer, ready for hw conversion (generates hls for the dataflow partition node)

### Tidy Up

In [20]:
from finn.util.visualization import showSrc, showInNetron
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.core.modelwrapper import ModelWrapper
from pathlib import Path
from config import IM_SIZE

# Setup Path
FINN_BIT_WIDTH = 8
onnx_path = f'../onnx/quant_model_{FINN_BIT_WIDTH}_bits.onnx'
onnx_name = Path(onnx_path).stem

model = ModelWrapper(onnx_path)

# Fix input size
#input_name = model.graph.input[0].name
#model.set_tensor_shape(input_name, [1, 3, *IM_SIZE])

# TIDY UP
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
tidy_path = f'./tidy_onnx/{onnx_name}_tidy.onnx'
Path(tidy_path).parent.mkdir(parents=True, exist_ok=True)
model.save(tidy_path)

showSrc(InferShapes)

/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/onnx.py:40: DeprecationWarning: `mapping.TENSOR_TYPE_TO_NP_TYPE` is now deprecated and will be removed in a future release.To silence this warning, please use `helper.tensor_dtype_to_np_dtype` instead.
  return np.zeros(dims, dtype=onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[vi.type.tensor_type.elem_type])


class InferShapes(Transformation):
    """Ensure every tensor in the model has a specified shape (ValueInfo)."""

    def apply(self, model):
        # hide your riches!
        hidden_ops = _hide_finn_ops(model)
        # call regular ONNX shape inference
        model = ModelWrapper(si.infer_shapes(model.model))
        # bring back hidden ops
        _restore_finn_ops(model, hidden_ops)
        return (model, False)



In [9]:
showInNetron(tidy_path)

Serving './tidy_onnx/quant_model_8_bits_tidy.onnx' at http://0.0.0.0:8081


### Pre processing

FINN model expects UINT8 input. According to Xilinx, this is highly beneficial for performance, because you can directly input raw data to the model, instead of relying on CPU for pre processing.

The the QFast-SCNN model exported to QONNX has the pre processing layers integrated in the Pytorch model, so the expected input is already from 0 to 255.

If the target model was trained with tensor inputs different than [0, 255], like the standard torch.Tensor [0, 1] or tensors with Imagenet normalization, you have two main options to follow:
* Modify your Pytorch model only for the QONNX export, integrating the pre processing inside the model (the option I have chosen for QFast-SCNN).
* Add the preprocessing layers in the QONNX model, following the "Adding Pre- and Postprocessing" section of Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook.

In [22]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
import torch
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN

# PRE PROC : NONE
model = ModelWrapper(tidy_path)

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

# Save the preprocessed model
preproc_path = f'./preproc_onnx/{onnx_name}_preproc.onnx'
Path(preproc_path).parent.mkdir(parents=True, exist_ok=True)
model.save(preproc_path)

In [23]:
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_shapes import InferShapes

# 1. Carrega o modelo
model = ModelWrapper(preproc_path)

# 2. Identifica a porta de entrada
input_name = model.graph.input[0].name

# 3. FORÇA A ÂNCORA MATEMÁTICA (Apenas inteiros permitidos)
print(f"Shape da entrada antes da âncora: {model.get_tensor_shape(input_name)}")
model.set_tensor_shape(input_name, [1, 3, 1024, 2048])
print(f"Shape da entrada depois da âncora: {model.get_tensor_shape(input_name)}")

# 4. Roda o InferShapes com a âncora fixada
model = model.transform(InferShapes())

# 5. Verifica se o primeiro nó e o último nó receberam as dimensões
first_node_out = model.graph.node[0].output[0]
last_node_out = model.graph.node[-1].output[0]

print(f"\nShape de saída do Primeiro Nó ({model.graph.node[0].op_type}): {model.get_tensor_shape(first_node_out)}")
print(f"Shape de saída do Último Nó ({model.graph.node[-1].op_type}): {model.get_tensor_shape(last_node_out)}")

Shape da entrada antes da âncora: [1, 3, 1024, 2048]
Shape da entrada depois da âncora: [1, 3, 1024, 2048]

Shape de saída do Primeiro Nó (Div): [1, 3, 1024, 2048]
Shape de saída do Último Nó (Conv): [1, 19, 128, 256]


In [33]:
import onnx
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
from qonnx.transformation.infer_shapes import InferShapes

model = ModelWrapper(preproc_path)

print("🔎 Forçando a estaticidade dos Inputs/Outputs globais...")

# 1. Trava o formato do Input Global (Substitui '?' ou 'batch_size' por 1)
input_name = model.graph.input[0].name
print(f" -> Travando Input Global '{input_name}' para [1, 3, 1024, 2048]")
model.set_tensor_shape(input_name, [1, 3, 1024, 2048])

# 2. Trava o formato do Output Global
output_name = model.graph.output[0].name
print(f" -> Travando Output Global '{output_name}' para [1, 19, 128, 256]")
model.set_tensor_shape(output_name, [1, 19, 128, 256])

# 3. Roda a inferência matemática com as bordas do modelo fixas
model = model.transform(InferShapes())

# 4. Varredura e correção de Inputs extras (ex: pesos que perderam metadados)
inputs_to_remove = []
for inp in model.graph.input:
    shape = model.get_tensor_shape(inp.name)
    # Se o shape for None OU contiver alguma string/dimensão dinâmica
    if shape is None or any(not isinstance(d, int) for d in shape):
        # Tenta resgatar as dimensões da matriz de inicialização (pesos estáticos)
        init = model.get_initializer(inp.name)
        if init is not None:
            print(f" -> [CORREÇÃO] Restaurando shape do peso '{inp.name}': {list(init.shape)}")
            model.set_tensor_shape(inp.name, list(init.shape))
        elif inp.name != input_name:
            print(f" -> [FANTASMA ENCONTRADO] O input global '{inp.name}' não tem utilidade. Marcando para deleção.")
            inputs_to_remove.append(inp)

# Deleta os fantasmas globais
for inp in inputs_to_remove:
    model.graph.input.remove(inp)

# 5. TESTE FINAL
if model.check_all_tensor_shapes_specified():
    print("\n✅ Sucesso! O compilador de hardware aceitou a arquitetura.")
    
    input_dict = {input_name: img_tensor.cpu().numpy()}
    output_dict = oxe.execute_onnx(model, input_dict)
    produced_qonnx = output_dict[list(output_dict.keys())[0]]
    
    print(f"🚀 INFERÊNCIA HARDWARE CONCLUÍDA! Shape: {produced_qonnx.shape}")
else:
    print("\n❌ Há algo muito obscuro. Mostrando a lista de tensores globais restantes:")
    for t in model.graph.input: print(f"Input: {t.name} -> {model.get_tensor_shape(t.name)}")
    for t in model.graph.output: print(f"Output: {t.name} -> {model.get_tensor_shape(t.name)}")

🔎 Forçando a estaticidade dos Inputs/Outputs globais...
 -> Travando Input Global 'global_in' para [1, 3, 1024, 2048]
 -> Travando Output Global 'global_out' para [1, 19, 128, 256]

❌ Há algo muito obscuro. Mostrando a lista de tensores globais restantes:
Input: global_in -> [1, 3, 1024, 2048]
Output: global_out -> [1, 19, 128, 256]


In [34]:
# Save the preprocessed model
preproc_path = f'./preproc_onnx/{onnx_name}_preproc.onnx'
Path(preproc_path).parent.mkdir(parents=True, exist_ok=True)
model.save(preproc_path)

In [4]:
showInNetron(preproc_path)

Serving './preproc_onnx/quant_model_8_bits_preproc.onnx' at http://0.0.0.0:8081


### Compare the Outputs of Pytorch Model and QONNX Model

Optional but highly recommended step. The Pytorch model will be loaded with the "finn" mode so the test input of this model is the same as the QONNX model.

In [11]:
import torch
from torchvision import transforms
from torchvision.datasets import Cityscapes
from my_finn_utils import load_state_dict, generate_cityscapes_labels, IdToTrainIdTransform
import models.QFastSCNN as qfscnn
from config import NUM_CLASSES, DATA_PATH

lable_conversion, id_names = generate_cityscapes_labels()

# Defining the Cityscapes validation dataset.
val_dataset = Cityscapes(
    root=DATA_PATH,
    split='val',
    mode='fine',
    target_type='semantic',
    transform=transforms.PILToTensor(), # Converting the PIL images to tensors, keeping the original pixel values (0-255) which is important for the quantized model that expects UINT8 inputs.
    target_transform=transforms.Compose([
        transforms.PILToTensor(), # Converting the PIL masks to tensors, keeping the original pixel values (0-255).
        IdToTrainIdTransform(lable_conversion), # Converting the original Cityscapes labels to the 19 classes used for training and evaluation, as per the Cityscapes benchmark.
    ])
)

# Importing the test image and mask
img_tensor, smnt_tensor = val_dataset[0]
img_tensor = img_tensor.unsqueeze(0) # Add batch dimension
print("Input image shape:", img_tensor.shape)
print("Input mask shape:", smnt_tensor.shape)

# Creating a Brevitas model instance and loading the quantized weights from the training phase.
brevitas_model = qfscnn.QFastSCNN(NUM_CLASSES, mode="finn")
brevitas_model = load_state_dict(brevitas_model, path="../train_environment/model_weights/quant_params/best_quant_model.pth", strict=False)
brevitas_model.eval();

Input image shape: torch.Size([1, 3, 1024, 2048])
Input mask shape: torch.Size([1, 1024, 2048])
Carregando modelo best_quant_model


In [12]:
import torch.nn.functional as F

# Run a foward pass on Brevitas model
with torch.inference_mode():
    brevitas_output = brevitas_model(img_tensor)
brevitas_output_upsampled = F.interpolate(brevitas_output, size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
brevitas_output_mask = torch.softmax(brevitas_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (brevitas_output_mask == smnt_tensor).sum().item()

print(f"Output shape from Brevitas model: {brevitas_output.shape}\n"
      f"Output shape after upsampling: {brevitas_output_upsampled.shape}\n"
      f"Output mask shape: {brevitas_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")

/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/usr/local/lib/python3.10/dist-packages/torch/overrides.py:1528: DeprecationWarning: Defining your `__torch_function__ as a plain method is deprecated and will be an error in future, please define it as a classmethod.
  warnings.warn("Defining your `__torch_function__ as a plain method is deprecated and "
/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment/models/QFastSCNN.py:193: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.cat([

Output shape from Brevitas model: torch.Size([1, 19, 128, 256])
Output shape after upsampling: torch.Size([1, 19, 1024, 2048])
Output mask shape: torch.Size([1, 1024, 2048])
Accuracy: 83.19%



In [13]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe

model = ModelWrapper(preproc_path)

# Run the QONNX inference.
input_tensor = img_tensor.detach().numpy()
input_dict = {"global_in": input_tensor}
output_dict = oxe.execute_onnx(model, input_dict)
produced_qonnx = output_dict[list(output_dict.keys())[0]]

Exception: Found unspecified tensor shapes, try infer_shapes

In [43]:
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe

model = ModelWrapper(preproc_path)

print("Injetando um formato fantasma no fio vazio ('')...")
# O HACK: Damos um formato [1] para o fio com nome de aspas vazias
model.set_tensor_shape("", [1])

# Agora perguntamos ao validador se ele está feliz
if model.check_all_tensor_shapes_specified():
    print("✅ BUG DO QONNX BYPASSADO! O validador aceitou o grafo.")
    
    input_name = model.graph.input[0].name
    input_dict = {input_name: img_tensor.detach().numpy()}
    
    print("Executando simulação matemática com os blocos Quant...")
    output_dict = oxe.execute_onnx(model, input_dict)
    produced_qonnx = output_dict[list(output_dict.keys())[0]]
    
    print(f"🚀 INFERÊNCIA QONNX CONCLUÍDA COM SUCESSO! Shape final: {produced_qonnx.shape}")
else:
    print("❌ O bypass falhou.")

Injetando um formato fantasma no fio vazio ('')...
✅ BUG DO QONNX BYPASSADO! O validador aceitou o grafo.
Executando simulação matemática com os blocos Quant...


InvalidArgument: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Unexpected input data type. Actual: (tensor(uint8)) , expected: (tensor(float))

In [17]:
model = ModelWrapper(tidy_path)
model.check_all_tensor_shapes_specified()

False

In [42]:
#model = ModelWrapper(preproc_path)
graph = model._model_proto.graph

for i in graph.node:
    print(f"Node: {i.name}, OpType: {i.op_type}, Inputs: {i.input}, Outputs: {i.output}")

Node: Div_0, OpType: Div, Inputs: ['global_in', 'Div_0_param0'], Outputs: ['Div_0_out0']
Node: Quant_0, OpType: Quant, Inputs: ['Quant_0_param0', 'Quant_0_param1', 'Quant_108_param1', 'Quant_108_param2'], Outputs: ['Quant_0_out0']
Node: Quant_1, OpType: Quant, Inputs: ['Quant_1_param0', 'Quant_1_param1', 'Quant_108_param1', 'Quant_108_param2'], Outputs: ['Quant_1_out0']
Node: Quant_2, OpType: Quant, Inputs: ['Quant_2_param0', 'Quant_2_param1', 'Quant_108_param1', 'Quant_108_param2'], Outputs: ['Quant_2_out0']
Node: Quant_3, OpType: Quant, Inputs: ['Quant_3_param0', 'Quant_3_param1', 'Quant_108_param1', 'Quant_108_param2'], Outputs: ['Quant_3_out0']
Node: Quant_4, OpType: Quant, Inputs: ['Quant_4_param0', 'Quant_4_param1', 'Quant_108_param1', 'Quant_108_param2'], Outputs: ['Quant_4_out0']
Node: Quant_5, OpType: Quant, Inputs: ['Quant_5_param0', 'Quant_5_param1', 'Quant_108_param1', 'Quant_108_param2'], Outputs: ['Quant_5_out0']
Node: Quant_6, OpType: Quant, Inputs: ['Quant_6_param0', 'Q

In [41]:
#model = ModelWrapper(preproc_path)

for i in graph.node:
    print(model.get_tensor_shape(i))
for o in graph.output:
    print(model.get_tensor_shape(o))

None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
